Imports + Paths

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [ ]:
import os
import sys
import random
from pathlib import Path
from tqdm import tqdm

BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "data", "synthetic_clean")

os.makedirs(DATA_DIR, exist_ok=True)

print("Base dir:", BASE_DIR)

Base dir: /home/dsi/skopavi/Project/genai_project/genai-project/avital_ssh


Load utils + classes

In [3]:
CODE_DIR = Path("./avital_ssh")

if str(CODE_DIR) not in sys.path:
    sys.path.append(str(CODE_DIR))

import utils

print("Classes:", utils.CLASS_NAMES)

Classes: ['clean', 'empty', 'finished_leftovers', 'full', 'unclassified']


Load prompts

In [4]:
prompts = utils.load_prompts()

for cls in prompts:
    print(cls, len(prompts[cls]))

clean 300
empty 300
finished_leftovers 300
full 300
unclassified 300


Config

In [5]:
NUM_IMAGES_PER_CLASS = 20

negative_prompt = """
multiple plates, more than one plate,
people, hands, utensils,
blurry, distorted, unrealistic,
extra objects, cluttered background
"""

MODEL SELECTOR

In [26]:
MODEL_NAME = "flux"   # "sd" / "sdxl" / "flux"

Load model

In [25]:
import torch

if 'pipe' in globals():
    del pipe

torch.cuda.empty_cache()

In [27]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

if MODEL_NAME == "sd":
    from diffusers import StableDiffusionPipeline

    pipe = StableDiffusionPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5"
    ).to(device)

    HEIGHT = 512
    WIDTH = 512
    STEPS = 40
    GUIDANCE = 8.5

    print("✅ SD loaded")

elif MODEL_NAME == "sdxl":
    from diffusers import StableDiffusionXLPipeline

    pipe = StableDiffusionXLPipeline.from_pretrained(
        "stabilityai/stable-diffusion-xl-base-1.0",
        torch_dtype=torch.float16
    ).to(device)

    HEIGHT = 768
    WIDTH = 768
    STEPS = 20
    GUIDANCE = 7.5

    print("✅ SDXL loaded")

elif MODEL_NAME == "flux":
    from diffusers import FluxPipeline

    pipe = FluxPipeline.from_pretrained(
        "black-forest-labs/FLUX.1-dev",
        torch_dtype=torch.float16
    ).to(device)

    pipe.enable_model_cpu_offload()

    HEIGHT = 512
    WIDTH = 512
    STEPS = 4
    GUIDANCE = 3.5

    print("⚠️ FLUX loaded (if memory survives 😅)")

else:
    raise ValueError("Invalid MODEL_NAME")

Loading pipeline components...:  29%|██▊       | 2/7 [00:02<00:07,  1.40s/it]


ValueError: Cannot instantiate this tokenizer from a slow version. If it's based on sentencepiece, make sure you have sentencepiece installed.

Output folders

In [9]:
MODEL_OUTPUT_DIR = os.path.join(DATA_DIR, MODEL_NAME)

for cls in utils.CLASS_NAMES:
    os.makedirs(os.path.join(MODEL_OUTPUT_DIR, cls), exist_ok=True)

print("Output dir:", MODEL_OUTPUT_DIR)

Output dir: /home/dsi/skopavi/Project/genai_project/genai-project/avital_ssh/data/model_outputs/sdxl


GENERATION LOOP

In [10]:
for cls in utils.CLASS_NAMES:
    print(f"\nGenerating {cls} with {MODEL_NAME}...")

    save_dir = os.path.join(MODEL_OUTPUT_DIR, cls)

    existing_files = os.listdir(save_dir)
    start_idx = len(existing_files)

    class_prompts = prompts[cls]

    selected_prompts = random.sample(
        class_prompts,
        min(len(class_prompts), NUM_IMAGES_PER_CLASS)
    )

    idx = start_idx

    for p in tqdm(selected_prompts):

        image = pipe(
            p,
            negative_prompt=negative_prompt,
            height=HEIGHT,
            width=WIDTH,
            num_inference_steps=STEPS,
            guidance_scale=GUIDANCE
        ).images[0]

        filename = f"{cls}_{idx}.png"
        image.save(os.path.join(save_dir, filename))

        idx += 1

    print(f"✅ Done {cls}")


Generating clean with sdxl...


100%|██████████| 20/20 [00:05<00:00,  3.76it/s]
/home/dsi/skopavi/Project/genai_project/genai-project/venv/lib64/python3.9/site-packages/diffusers/pipelines/stable_diffusion_xl/pipeline_stable_diffusion_xl.py:748: FutureWarning: `upcast_vae` is deprecated and will be removed in version 1.0.0. `upcast_vae` is deprecated. Please use `pipe.vae.to(torch.float32)`. For more details, please refer to: https://github.com/huggingface/diffusers/pull/12619#issue-3606633695.
  deprecate(
100%|██████████| 20/20 [01:45<00:00,  5.25s/it]


✅ Done clean

Generating empty with sdxl...


100%|██████████| 20/20 [02:38<00:00,  7.91s/it]


✅ Done empty

Generating finished_leftovers with sdxl...


...

✅ Done finished_leftovers

Generating full with sdxl...


100%|██████████| 20/20 [02:27<00:00,  7.38s/it]


✅ Done full

Generating unclassified with sdxl...


100%|██████████| 20/20 [02:26<00:00,  7.31s/it]

✅ Done unclassified


Count images

In [13]:
print("\n=== SUMMARY ===")

for cls in utils.CLASS_NAMES:
    path = os.path.join(MODEL_OUTPUT_DIR, cls)
    print(MODEL_OUTPUT_DIR)
    print(f"{cls}: {len(os.listdir(path))} images")


=== SUMMARY ===
/home/dsi/skopavi/Project/genai_project/genai-project/avital_ssh/data/model_outputs/sd
clean: 40 images
/home/dsi/skopavi/Project/genai_project/genai-project/avital_ssh/data/model_outputs/sd
empty: 22 images
/home/dsi/skopavi/Project/genai_project/genai-project/avital_ssh/data/model_outputs/sd
finished_leftovers: 20 images
/home/dsi/skopavi/Project/genai_project/genai-project/avital_ssh/data/model_outputs/sd
full: 20 images
/home/dsi/skopavi/Project/genai_project/genai-project/avital_ssh/data/model_outputs/sd
unclassified: 20 images
